In [ ]:
# 1. IMPORTAÇÃO DAS BIBLIOTECAS
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import joblib # Biblioteca para salvar o modelo treinado

# 2. CARREGAMENTO DOS DADOS (Substitua pelo seu caminho se necessário)
caminho = r'C:\Users\mauro.pupim\OneDrive - CantuStore\Documentos\Cursos\FIAP\Tech Challenge - Fase 5\data\BASE DE DADOS PEDE 2024 - DATATHON.xlsx'

# Vamos focar na base mais recente (2024) ou empilhar todas. 
# Para simplificar e ter os dados mais atualizados da jornada, usaremos a de 2024.
df_24 = pd.read_excel(caminho, sheet_name='PEDE2024')

# 3. SELEÇÃO CIRÚRGICA DE FEATURES
indicadores = ['IAN', 'IDA', 'IEG', 'IAA', 'IPS', 'IPP', 'INDE']
colunas_selecionadas = []

# Procura as colunas exatas (com ou sem o " 2024") para evitar pegar lixo
for ind in indicadores:
    if f"{ind} 2024" in df_24.columns:
        colunas_selecionadas.append(f"{ind} 2024")
    elif ind in df_24.columns:
        colunas_selecionadas.append(ind)

df_ml = df_24[colunas_selecionadas].copy()

# Renomeia para padronizar (Tira o " 2024" se existir)
df_ml.columns = [col.replace(' 2024', '').strip() for col in df_ml.columns]

# A MÁGICA CONTRA O ERRO: Remove qualquer coluna duplicada que tenha passado
df_ml = df_ml.loc[:, ~df_ml.columns.duplicated()]

# 4. LIMPEZA DOS DADOS PARA O MODELO
for col in df_ml.columns:
    # Como garantimos que não há duplicatas, o .str vai funcionar perfeitamente agora
    df_ml[col] = df_ml[col].astype(str).str.replace(',', '.')
    df_ml[col] = pd.to_numeric(df_ml[col], errors='coerce')

# Remove os alunos que estão com as notas vazias (NaN)
df_ml = df_ml.dropna()

# ----------------------------------------------------------------------
# ABA 5: MODELO PREDITIVO (REAL)
# ----------------------------------------------------------------------
with aba5:
    st.markdown("### 🤖 Previsão de Risco de Defasagem (Machine Learning)")
    st.write("Insira os indicadores do aluno para que o modelo preditivo calcule a probabilidade de queda de rendimento e defasagem escolar.")
    
    # Carrega o modelo treinado (usamos cache para não ler o arquivo do zero a cada clique)
    @st.cache_resource
    def carregar_modelo():
        try:
            return joblib.load('modelo_passos_magicos.pkl')
        except:
            return None
            
    modelo = carregar_modelo()
    
    if modelo is None:
        st.error("⚠️ Arquivo 'modelo_passos_magicos.pkl' não encontrado. Certifique-se de que ele está na mesma pasta que o app.py.")
    else:
        # Formulário para coletar os dados na tela
        with st.form("form_ml"):
            st.subheader("Indicadores do Aluno")
            col_ml1, col_ml2, col_ml3 = st.columns(3)

            with col_ml1:
                # Se o usuário filtrou alguma fase no menu lateral, usa ela. Se não, mostra todas.
                opcoes_fase = fases_selecionadas if fases_selecionadas else ["Fase 1", "Fase 2", "Fase 3", "Fase 4", "Fase 5", "Fase 6", "Fase 7", "Fase 8"]
                fase_input = st.selectbox("Fase do Aluno na ONG", opcoes_fase)
                
                ida_input = st.slider("Desempenho Acadêmico (IDA)", 0.0, 10.0, 5.0)
                ieg_input = st.slider("Engajamento nas Atividades (IEG)", 0.0, 10.0, 5.0)

            with col_ml2:
                ian_input = st.slider("Adequação de Nível (IAN)", 0.0, 10.0, 5.0)
                ips_input = st.slider("Aspecto Psicossocial (IPS)", 0.0, 10.0, 5.0)
                ipp_input = st.slider("Aspecto Psicopedagógico (IPP)", 0.0, 10.0, 5.0)

            with col_ml3:
                iaa_input = st.slider("Autoavaliação (IAA)", 0.0, 10.0, 5.0)
                inde_input = st.slider("Índice de Desenvolvimento (INDE)", 0.0, 10.0, 5.0)

            submit_button = st.form_submit_button("Gerar Análise Preditiva", use_container_width=True)

        # A mágica acontece quando você clica no botão
        if submit_button:
            with st.spinner('Processando dados na Random Forest...'):
                time.sleep(1) # Leve delay para efeito visual
                
                # 1. Organiza os dados de entrada EXATAMENTE na mesma ordem que o modelo foi treinado
                dados_entrada = pd.DataFrame([[ian_input, ida_input, ieg_input, iaa_input, ips_input, ipp_input, inde_input]], 
                                             columns=['IAN', 'IDA', 'IEG', 'IAA', 'IPS', 'IPP', 'INDE'])
                
                # 2. Faz a previsão de probabilidade (Retorna um array com chance de [0, 1])
                # A posição [0][1] pega a probabilidade de ser classe 1 (Risco)
                probabilidade = modelo.predict_proba(dados_entrada)[0][1]
                risco = probabilidade * 100 

                st.markdown("---")
                st.subheader("Resultado do Modelo Random Forest")

                # Exibe o alerta com base no cálculo do modelo real
                if risco >= 60:
                    st.error(f"🚨 **ALTO RISCO DETECTADO!** Probabilidade de defasagem: **{risco:.1f}%**")
                    st.write("**Ação sugerida:** O perfil validado pelo histórico da ONG indica um forte padrão de queda. O aluno necessita de intervenção imediata.")
                elif risco >= 30:
                    st.warning(f"⚠️ **RISCO MODERADO.** Probabilidade de defasagem: **{risco:.1f}%**")
                    st.write("**Ação sugerida:** Sinais de alerta. Monitorar de perto o Engajamento e o aspecto Psicopedagógico nas próximas semanas.")
                else:
                    st.success(f"✅ **BAIXO RISCO.** Probabilidade de defasagem: **{risco:.1f}%**")
                    st.write("**Ação sugerida:** Aluno com padrão saudável e resiliente. Manter o acompanhamento padrão na fase.")

# 6. SEPARAÇÃO DE TREINO E TESTE (80% para ensinar, 20% para testar)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 7. TREINAMENTO DO MODELO (Machine Learning)
modelo_rf = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')
modelo_rf.fit(X_train, y_train)

# 8. AVALIAÇÃO DO MODELO
previsoes = modelo_rf.predict(X_test)
print("=== AVALIAÇÃO DO MODELO PREDITIVO ===")
print(f"Acurácia: {accuracy_score(y_test, previsoes) * 100:.2f}%\n")
print("Relatório de Classificação:")
print(classification_report(y_test, previsoes))

# 9. EXPORTAÇÃO (DEPLOY DO MODELO)
# Isso vai criar um arquivo .pkl na sua pasta. É esse arquivo que o Streamlit vai ler!
caminho_salvar = r'C:\Users\mauro.pupim\OneDrive - CantuStore\Documentos\Cursos\FIAP\Tech Challenge - Fase 5\models\modelo_passos_magicos.pkl'
joblib.dump(modelo_rf, caminho_salvar)
print(f"✅ Modelo salvo com sucesso DIRETAMENTE na pasta models!")